# QCircuitHEOM-code

In [ ]:
%load_ext autoreload
%autoreload 2

import os
ROOT_DIR = os.getcwd()[:os.getcwd().rfind('HEOM')]+ 'HEOM'
os.chdir(ROOT_DIR)

from qcheom import *
plt.style.use('qcheom-default')

import numpy as np 
import qutip as q
import matplotlib.pyplot as plt
import pandas as pd
import time

red, blue, orange, grey = '#B74244', '#459DD9', '#ED8532', '#A9ACBC'

## Quickstart (for readme and docs)

In [ ]:
from qiskit import QuantumCircuit
from qcheom import calcTimeEvo

qc = QuantumCircuit(1)
qc.h(0)

# expected runtime: 1 min
calcTimeEvo(
   fileName="result",
   qc=qc,
   numQ=1,
   freqQ=[5.0],          # GHz
   rhoIni=[[1,0],[0,0]],
   gateTime=[0.16],      # ns
   idlingTime=0.01,       # ns
   T=30,                 # mK
   T1=32,                # µs
   omegaC=20,
   exp=1/8,
   tol=1e-6,
   dtFB=0.1,             # ps
   depth=[1],
   bondDim=5,
   strideTime=0.01,       # ns
)

## Minimal example

In [ ]:
import numpy as np
from qiskit import QuantumCircuit
from qcheom import calcTimeEvo

qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)

calcTimeEvo(
    qc=qc, numQ=2, fileName="H_CNOT",
    # system
    freqQ=[5, 5], rhoIni=np.diag([1, 0, 0, 0]),
    gateTime=[16, 16, 50], idlingTime=1,
    # bath
    T=30, T1=32, omegaC=20, exp=1/8,
    # numerics
    tol=1e-6, depth=[1, 1], bondDim=5,
    dtFB=0.1, strideTime=0.1,
)

In [ ]:
import numpy as np
from qiskit import QuantumCircuit
from qcheom import calcTimeEvo

qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)

calcTimeEvo(
    qc=qc, numQ=2, fileName="H_CNOT",
    # system
    freqQ=[5, 5], rhoIni=np.diag([1, 0, 0, 0]),
    gateTime=[0.16, 0.16, 0.50], idlingTime=0.01,
    # bath
    T=30, T1=32, omegaC=20, exp=1/8,
    # numerics
    tol=1e-6, depth=[1, 1], bondDim=5,
    dtFB=0.1, strideTime=0.001,
)

## Appendix

In [ ]:
import numpy as np
from qiskit import QuantumCircuit
from qcheom import calcTimeEvo

spectra = {"s1": 1, "s2": 1/2, "s8": 1/8}

In [ ]:
qc = QuantumCircuit(1)
for _ in range(3):
    qc.rx(np.pi, 0)
    qc.delay(0, 0)               # keeps the two pi pulses separate
    qc.rx(np.pi, 0)
    qc.delay(100000, 0)          # 10 ns idle interval (in steps of dtFB)

for name, s in spectra.items():
    calcTimeEvo(qc=qc, numQ=1, fileName=f"3Rx_{name}",
                freqQ=[5], rhoIni=np.diag([1, 0]), gateTime=[16], idlingTime=0,
                T=30, T1=10, omegaC=20, exp=s,
                tol=1e-6, depth=[1], bondDim=5, dtFB=0.1, strideTime=0.1)

In [ ]:
def pulse(qc, axis):
    qc.delay(100000, 0)          # 10 ns idle interval before each pulse
    getattr(qc, axis)(np.pi, 0)

xy4 = QuantumCircuit(1)
for axis in ["rx", "ry", "rx", "ry"]:
    pulse(xy4, axis)

xy8 = QuantumCircuit(1)
for axis in ["rx", "ry", "rx", "ry", "ry", "rx", "ry", "rx"]:
    pulse(xy8, axis)

for label, qc in [("XY4", xy4), ("XY8", xy8)]:
    for name, s in spectra.items():
        calcTimeEvo(qc=qc, numQ=1, fileName=f"{label}_{name}",
                    freqQ=[5], rhoIni=np.full((2, 2), 0.5), gateTime=[16],
                    idlingTime=0, T=30, T1=32, omegaC=50, exp=s,
                    tol=1e-6, depth=[1], bondDim=5, dtFB=0.1, strideTime=0.1)

In [ ]:
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)

for name, s in spectra.items():
    calcTimeEvo(qc=qc, numQ=2, fileName=f"Bell_{name}",
                freqQ=[5, 5], rhoIni=np.diag([1, 0, 0, 0]), gateTime=[16, 16, 50],
                idlingTime=1, T=30, T1=1, omegaC=20, exp=s,
                tol=1e-6, depth=[1, 1], bondDim=5, dtFB=0.1, strideTime=0.1)

In [ ]:
qc = QuantumCircuit(3)
qc.h(0)
qc.cx(0, 1)
qc.cx(1, 2)

for name, s in spectra.items():
    calcTimeEvo(qc=qc, numQ=3, fileName=f"GHZ_{name}",
                freqQ=[5, 5, 5], rhoIni=np.diag([1, 0, 0, 0, 0, 0, 0, 0]),
                gateTime=[16, 16, 16, 50, 50], idlingTime=0,
                T=30, T1=32, omegaC=20, exp=s,
                tol=1e-6, depth=[1, 1, 1], bondDim=5, dtFB=0.2, strideTime=0.1)